# (Homework) Week 6 - DataScience Bootcamp Fall 2025

All solution cells are replaced with `# TODO` placeholders so you can fill them in.

**Name:** \
**Email:**

---

### Problem 1: Dataset Splitting

1. You have recordings of 44 phones from 100 people; each person records ~200 phones/day for 5 days.
   - Design a valid training/validation/test split strategy that ensures the model generalizes to **new speakers**.

2. You now receive an additional dataset of 10,000 phone recordings from **Kilian**, a single speaker.
   - You must train a model that performs well **specifically for Kilian**, while also maintaining generalization.

*Describe your proposed split strategy and reasoning.* (Theory)

In [ ]:
# Solution to Problem 1: Dataset Splitting (theory answer printed)

proposal = '''
1) Original dataset (44 phones × 100 people, ~200 phones/day for 5 days):
   - Split by speaker, not by recording. Use a speaker-based holdout to ensure
     generalization to new speakers.
   - Example split:
     * Train: 70% of speakers (70 people) — use all recordings from those speakers
     * Validation: 15% of speakers (15 people)
     * Test: 15% of speakers (15 people)
   - If per-speaker variability is large across days, ensure that recordings
     from all days are represented for each speaker within the assigned split.

   Rationale: Splitting by recording (randomly) could leak speaker-specific
   features between train and test; splitting by speaker prevents this.

2) Additional dataset of 10,000 recordings from Kilian (single speaker):
   - If the goal is to build a model that performs well specifically for
     Kilian while retaining generalization, use a two-part strategy:
     * Keep the original train/val/test split by speakers for generalization.
     * Create a Kilian-specific fine-tuning split: e.g., use 80% of Kilian's
       recordings to fine-tune (or adapt) the model, 10% for Kilian-val, 10%
       for Kilian-test. Evaluate both: general test (original test set) and
       Kilian-test to measure specialized performance.
   - Optionally, use transfer learning / domain-adaptation: train base model
     on multi-speaker train set, then fine-tune (or adapt) on Kilian's data.

   Rationale: This preserves generalization (evaluated on original test speakers)
   while allowing targeted performance improvement for Kilian.
'''

print(proposal)

### Problem 2: K-Nearest Neighbors

1. **1-NN Classification:** Given dataset:

   Positive: (1,2), (1,4), (5,4)

   Negative: (3,1), (3,2)

   Plot the 1-NN decision boundary and classify new points visually.

2. **Feature Scaling:** Consider dataset:

   Positive: (100,2), (100,4), (500,4)

   Negative: (300,1), (300,2)

   What would the 1-NN classify point (500,1) as **before and after scaling** to [0,1] per feature?

3. **Handling Missing Values:** How can you modify K-NN to handle missing features in a test point?

4. **High-dimensional Data:** Why can K-NN still work well for images even with thousands of pixels?


In [ ]:
# Solution to Problem 2: K-Nearest Neighbors (plots + scaling + handling missing values)
import numpy as np
import matplotlib.pyplot as plt

# Part 1: 1-NN decision boundary for small 2D dataset
pos = np.array([[1,2],[1,4],[5,4]])
neg = np.array([[3,1],[3,2]])
X = np.vstack([pos, neg])
y = np.array([1]*len(pos) + [-1]*len(neg))

# simple 1-NN classifier using numpy
def knn_predict(train_X, train_y, query, k=1):
    dists = np.linalg.norm(train_X - query, axis=1)
    idx = np.argsort(dists)[:k]
    votes = train_y[idx]
    return np.sign(votes.sum()) if votes.sum()!=0 else votes[0]

# Plot decision boundary by grid
xx, yy = np.meshgrid(np.linspace(0,6,300), np.linspace(0,5,300))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = np.array([knn_predict(X, y, q, k=1) for q in grid])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(6,4))
plt.contourf(xx, yy, Z, alpha=0.2, levels=[-1,0,1], colors=['#ffdddd','#ddffdd'])
plt.scatter(pos[:,0], pos[:,1], color='g', label='Positive')
plt.scatter(neg[:,0], neg[:,1], color='r', label='Negative')
plt.title('1-NN decision boundary (k=1)')
plt.legend()
plt.show()

# Part 2: Feature scaling effect
pos2 = np.array([[100,2],[100,4],[500,4]])
neg2 = np.array([[300,1],[300,2]])
X2 = np.vstack([pos2, neg2])
y2 = np.array([1]*len(pos2) + [-1]*len(neg2))
query = np.array([500,1])

# Predict before scaling
pred_before = knn_predict(X2, y2, query, k=1)

# Scale each feature to [0,1]
mins = X2.min(axis=0)
maxs = X2.max(axis=0)
X2_scaled = (X2 - mins) / (maxs - mins)
query_scaled = (query - mins) / (maxs - mins)
pred_after = knn_predict(X2_scaled, y2, query_scaled, k=1)

print('Prediction before scaling for (500,1):', 'Positive' if pred_before==1 else 'Negative')
print('Prediction after scaling for (500,1):', 'Positive' if pred_after==1 else 'Negative')

# Part 3: Handling missing features in a test point
print('\nHandling missing values (strategy):')
strategy = '''
- When a test point has missing features, compute distances only over the
  subset of available features and normalize the distance by the number of
  used features (or use feature-wise imputation).
- Alternatively, use weighted distance where missing features contribute
  zero but adjust vote thresholds accordingly.
'''
print(strategy)

# Demonstration of distance-on-available-features
train = np.array([[1,2,3],[4,5,6],[7,8,9]])
labels = np.array([1,-1,1])
query_missing = np.array([np.nan, 5, 2])

def knn_with_missing(train_X, train_y, query, k=1):
    mask = ~np.isnan(query)
    sub_train = train_X[:, mask]
    sub_query = query[mask]
    dists = np.linalg.norm(sub_train - sub_query, axis=1)
    idx = np.argsort(dists)[:k]
    votes = train_y[idx]
    return np.sign(votes.sum())

print('Example knn with missing ->', knn_with_missing(train, labels, query_missing))

# Part 4: High-dimensional data intuition (short answer)
print('\nHigh-dimensional intuition:')
print('K-NN can work for images because nearby images (in pixel space) often')
print('share semantic structure; using appropriate distance/feature transforms (')
print('PCA, deep features) reduces dimensionality and helps K-NN perform well.')

### Problem 3: Part 1

You are given a fully trained Perceptron model with weight vector **w**, along with training set **D_TR** and test set **D_TE**.

1. Your co-worker suggests evaluating $h(x) = sign(w \cdot x)$ for every $(x, y)$ in D_TR and D_TE. Does this help determine whether test error is higher than training error?
2. Why is there no need to compute training error explicitly for the Perceptron algorithm?

In [ ]:
# Solution to Problem 3 (Part 1): Theory answers printed

answer = '''
1) Evaluating h(x)=sign(w·x) on D_TR and D_TE and comparing errors can tell you
   whether the classifier's empirical error differs between train and test, but
   simply running the sign on those sets is just computing error rates — to
   conclude that test error is higher you must compare the two error rates and
   consider variance and sample sizes. In short: yes, evaluating h on both
   sets gives the measured train and test errors; determining whether test
   error is "higher" requires comparing these numbers and understanding
   statistical significance.

2) For the Perceptron algorithm, when training to convergence on a separable
   dataset, the final perceptron weight vector classifies the training set
   perfectly (zero training error). If training didn't converge (non-separable)
   you typically track updates or use a maximum epoch; but standard perceptron
   theory implies that after convergence training error is zero, so explicitly
   computing training error is unnecessary for that conclusion.
'''

print(answer)

### Problem 3: Two-point 2D Dataset (Part 2)

Run the Perceptron algorithm **by hand or in code** on the following data:

1. Positive class: (10, -2)
2. Negative class: (12, 2)

Start with $w_0 = (0, 0)$ and a learning rate of 1.

- Compute how many updates are required until convergence.
- Write down the sequence of $w_i$ vectors.

In [ ]:
# Solution to Problem 3 (Part 2): Run Perceptron on two 2D points until convergence
import numpy as np

# Data: Positive (10, -2) label +1; Negative (12, 2) label -1
X = np.array([[10.0, -2.0],[12.0, 2.0]])
y = np.array([1.0, -1.0])

w = np.zeros(2)
lr = 1.0
updates = 0
w_history = [w.copy()]
max_iters = 1000

for epoch in range(max_iters):
    changed = False
    for xi, yi in zip(X, y):
        if np.sign(np.dot(w, xi)) != yi:
            w = w + lr * yi * xi
            updates += 1
            w_history.append(w.copy())
            changed = True
    if not changed:
        break

print('Converged after updates:', updates)
print('Sequence of w vectors:')
for i, ww in enumerate(w_history):
    print(f'w_{i}:', ww)

# Also show final decision on the two points
print('\nFinal classification:')
for xi, yi in zip(X, y):
    print('x=', xi, 'true y=', yi, 'pred=', np.sign(np.dot(w, xi)))

### Problem 4: Reconstructing the Weight Vector

Given the log of Perceptron updates:

| x | y | count |
|---|---|--------|
| (0, 0, 0, 0, 4) | +1 | 2 |
| (0, 0, 6, 5, 0) | +1 | 1 |
| (3, 0, 0, 0, 0) | -1 | 1 |
| (0, 9, 3, 6, 0) | -1 | 1 |
| (0, 1, 0, 2, 5) | -1 | 1 |

Assume learning rate = 1 and initial weight $w_0 = (0, 0, 0, 0, 0)$.

Compute the final weight vector after all updates.

In [ ]:
# Solution to Problem 4: Reconstructing the Weight Vector
import numpy as np

# Logged perceptron updates: (x, y, count)
updates = [
    (np.array([0, 0, 0, 0, 4]), +1, 2),
    (np.array([0, 0, 6, 5, 0]), +1, 1),
    (np.array([3, 0, 0, 0, 0]), -1, 1),
    (np.array([0, 9, 3, 6, 0]), -1, 1),
    (np.array([0, 1, 0, 2, 5]), -1, 1),
]

w = np.zeros(5, dtype=int)
print("Initial w:", w)

# Apply updates (learning rate = 1)
for x, y, count in updates:
    contrib = y * x * count
    print(f"Applying update: y={y}, count={count}, x={x}, contribution={contrib}")
    w += contrib
    print("Intermediate w:", w)

print("\nFinal weight vector w:", w)

### Problem 5: Visualizing Perceptron Convergence

Implement a Perceptron on a small 2D dataset with positive and negative examples.

- Plot the data points.
- After each update, visualize the decision boundary.
- Show how it converges to a stable separator.

In [ ]:
# Solution to Problem 5: Visualizing Perceptron Convergence (2D)
import numpy as np
import matplotlib.pyplot as plt

# Simple 2D dataset (linearly separable)
pos = np.array([[2,3],[3,3],[3,2],[4,3]])
neg = np.array([[0,0],[1,0],[0,1],[1,1]])
X = np.vstack([pos, neg])
y = np.array([1]*len(pos) + [-1]*len(neg))

# Perceptron training, record weight after each update
w = np.zeros(3)  # augmented weight [w0, w1, b]
lr = 1.0
w_history = [w.copy()]

# augment inputs with bias term
X_aug = np.hstack([X, np.ones((X.shape[0],1))])

max_epochs = 20
for epoch in range(max_epochs):
    updated = False
    for xi, yi in zip(X_aug, y):
        if np.sign(np.dot(w, xi)) != yi:
            w = w + lr * yi * xi
            w_history.append(w.copy())
            updated = True
    if not updated:
        break

# Plot sequence of decision boundaries
n = len(w_history)
cols = min(4, n)
rows = (n + cols - 1) // cols
fig, axs = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
axs = np.array(axs).reshape(-1)
xx = np.linspace(-1, 5, 200)

for i, ww in enumerate(w_history):
    ax = axs[i]
    # decision boundary: w0*x + w1*y + b = 0 => y = -(w0/w1)*x - b/w1
    if abs(ww[1]) > 1e-6:
        yy = -(ww[0]/ww[1]) * xx - (ww[2]/ww[1])
        ax.plot(xx, yy, '-k')
    else:
        # near-vertical decision boundary
        xline = -ww[2]/ww[0] if abs(ww[0])>1e-6 else 0
        ax.axvline(xline, color='k')
    ax.scatter(pos[:,0], pos[:,1], color='g', label='pos')
    ax.scatter(neg[:,0], neg[:,1], color='r', label='neg')
    ax.set_title(f'After update {i}')
    ax.set_xlim(-1,5)
    ax.set_ylim(-1,5)
    ax.legend()

# hide unused subplots
for j in range(n, len(axs)):
    axs[j].axis('off')

plt.tight_layout()
plt.show()

print('Perceptron converged in', len(w_history)-1, 'updates (excluding initial w).')
print('Final weight vector (augmented):', w_history[-1])